# RoadGuard — Model Training and Evaluation

Run the configured **special LOFO** experiment: RGB+depth teacher, supervised RGB baseline, and distilled RGB student. The training implementation and checks live in `roadguard`, not in this notebook.

## 1. Mount Drive and install RoadGuard

Use a GPU Colab runtime. The repository must contain the validated `experiment.py` and updated depth-compatible `dataset.py`. Clone the repository if needed, or pull the committed updates before proceeding.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os
import subprocess
import base64

# Mount Google Drive
drive.mount("/content/drive")

# Clone private RoadGuard repository
REPO = Path("/content/road-guard")

if not REPO.exists():
    token = userdata.get("GITHUB_TOKEN")

    credentials = base64.b64encode(
        f"x-access-token:{token}".encode()
    ).decode()

    env = os.environ.copy()
    env["GIT_CONFIG_COUNT"] = "1"
    env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {credentials}"

    subprocess.run(
        ["git", "clone", "https://github.com/RanaShoaaib/RoadGuard.git", str(REPO)],
        check=True,
        env=env,
    )

    del token, credentials, env

# Install RoadGuard
os.chdir(REPO)
subprocess.run(["python", "-m", "pip", "install", "-e", "."], check=True)

print("Repository:", REPO)

In [ ]:
import sys
sys.path.insert(0, str(REPO / "src"))

# Verify the import before starting training.
from roadguard.training.experiment import run_experiment
print("RoadGuard imported successfully")

## 2. Stage the prepared dataset locally

Notebook 1 saved one TAR package per archive in `RoadGuard_Processed/towns/` and the manifest and LOFO splits in `RoadGuard_Processed/processed/`. Unpacking them to the Colab disk avoids repeated Google Drive reads during training. Allow sufficient local disk space for the unpacked files.

In [ ]:
from roadguard.data.preparation import stage_prepared_dataset

DRIVE_PREPARED = Path("/content/drive/MyDrive/RoadGuard_Processed")
STAGE = Path("/content/roadguard_data")

staged = stage_prepared_dataset(DRIVE_PREPARED, STAGE)
DATA_ROOT = staged["data_root"]
FOLD_PATH = staged["processed_dir"] / "lofo_splits" / "special.csv"

if not FOLD_PATH.is_file():
    raise FileNotFoundError(f"Missing special LOFO split: {FOLD_PATH}")
print("Dataset staged:", DATA_ROOT)
print("LOFO fold:", FOLD_PATH)

## 3. Run the configured experiment

The YAML controls architecture, epochs, optimizer, teacher depth perturbations, and distillation settings. The runner selects best checkpoints by supervised validation loss and evaluates the held-out test set after training. Outputs are saved persistently to Drive. **An interrupted run cannot currently resume**; the runner refuses to overwrite a nonempty run directory.

In [ ]:
import torch
from roadguard.training.config import load_config
from roadguard.training.experiment import run_experiment

# EXPERIMENT CONFIGURATION
# Change the filename below to select the architecture and loss.
# Available configurations:
#   unet_bce.yaml
#   unet_focal.yaml
#   segformer_bce.yaml
#   segformer_focal.yaml
CONFIG_PATH = REPO / "configs" / "unet_bce.yaml"

OUTPUT_ROOT = Path("/content/drive/MyDrive/RoadGuard_Experiments")
config = load_config(CONFIG_PATH)
assert config["experiment"]["lofo_family"] == "special"
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before training")
print("GPU:", torch.cuda.get_device_name(0))
print("Experiment:", config["experiment"])

results = run_experiment(
    config_path=CONFIG_PATH,
    fold_path=FOLD_PATH,
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    device="cuda",
)


## 4. Compare held-out test performance

The primary comparison is **pixel AUPRC**. These are pixel-level metrics aggregated over all frames of the held-out `special` scenarios, including frames where the anomaly is not visible.

In [ ]:
import pandas as pd

comparison = pd.DataFrame.from_dict(results, orient="index")
comparison.index.name = "model"
cols = ["pixel_auprc", "pixel_auroc", "dice", "iou", "precision", "recall"]
display(comparison[cols].round(4))

run_dir = OUTPUT_ROOT / config["experiment"]["name"] / "special" / f"seed_{config['experiment']['seed']}"
print("Saved experiment results:", run_dir)
